In [32]:
#TESTE DO LOGIN: FUNCIONOU
import requests
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widgets para login
user_input = widgets.Text(
    value='',
    description='Usuário:',
    placeholder='Digite o usuário'
)
password_input = widgets.Password(
    description='Senha:',
    placeholder='Digite a senha'
)
login_button = widgets.Button(
    description='Logar',
    button_style='warning'
)
test_login_button = widgets.Button(
    description='Testar Login',
    button_style='success'
)
output = widgets.Output()

# Armazenamento seguro em memória
secure_credentials = {}

def on_login_clicked(b):
    secure_credentials['user'] = user_input.value.strip()
    secure_credentials['password'] = password_input.value.strip()
    user_input.value = ''
    password_input.value = ''
    login_button.description = 'Dados salvos, clique em Testar Login'
    login_button.button_style = 'info'
    with output:
        clear_output()
        print("Credenciais armazenadas com sucesso.")

def testar_login(b):
    if 'user' not in secure_credentials or 'password' not in secure_credentials:
        with output:
            clear_output()
            print("Erro: por favor insira usuário e senha e clique em Logar antes.")
        return
    
    url = "https://sigen.cidasc.sc.gov.br/Account/Login"
    payload = {
        "nmUsuario": secure_credentials['user'],
        "dsSenha": secure_credentials['password'],
        "dsBrowser": "Chrome",
        "dsVersion": "136"
    }
    headers = {
        "Content-Type": "application/json;charset=UTF-8",
        "Accept": "application/json, text/plain, */*",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
        "Origin": "https://sigen.cidasc.sc.gov.br",
        "Referer": "https://sigen.cidasc.sc.gov.br/Account/LogOn"
    }
    
    with output:
        clear_output()
        print("Enviando login...")
    
    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        with output:
            print(f"Status Code: {response.status_code}")
            print("Resposta (raw):", response.text)
            try:
                resposta_json = response.json()
                print("Resposta JSON formatada:")
                print(json.dumps(resposta_json, indent=2, ensure_ascii=False))
                if response.status_code == 200 and 'redirectUrl' in resposta_json:
                    # Salva cookies para uso posterior
                    secure_credentials['cookies'] = response.cookies.get_dict()
                    print("Login bem-sucedido. Cookies armazenados para requisições futuras.")
                else:
                    print("Falha no login: verifique usuário/senha.")
            except Exception as e:
                print("Não foi possível interpretar a resposta JSON:", e)
    except Exception as e:
        with output:
            print("Erro na requisição:", e)

# Associa botões às funções
login_button.on_click(on_login_clicked)
test_login_button.on_click(testar_login)

# Layout
login_box = widgets.VBox([user_input, password_input, login_button, test_login_button, output])
display(login_box)





In [ ]:
#LOGIN SEM TESTE

import requests
import json
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widgets para login
user_input = widgets.Text(
    value='',
    description='Usuário:',
    placeholder='Digite o usuário'
)
password_input = widgets.Password(
    description='Senha:',
    placeholder='Digite a senha'
)
login_button = widgets.Button(
    description='Logar',
    button_style='warning'
)
output = widgets.Output()

# Armazenamento seguro em memória
secure_credentials = {}

def on_login_clicked(b):
    # Pega as credenciais
    user = user_input.value.strip()
    password = password_input.value.strip()
    if not user or not password:
        with output:
            clear_output()
            print("Por favor, preencha usuário e senha.")
        return
    
    # Limpa os campos para segurança
    user_input.value = ''
    password_input.value = ''
    
    url = "https://sigen.cidasc.sc.gov.br/Account/Login"
    payload = {
        "nmUsuario": user,
        "dsSenha": password,
        "dsBrowser": "Chrome",
        "dsVersion": "136"
    }
    headers = {
        "Content-Type": "application/json;charset=UTF-8",
        "Accept": "application/json, text/plain, */*",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
        "Origin": "https://sigen.cidasc.sc.gov.br",
        "Referer": "https://sigen.cidasc.sc.gov.br/Account/LogOn"
    }
    
    with output:
        clear_output()
        print("Enviando login...")
    
    try:
        response = requests.post(url, headers=headers, data=json.dumps(payload))
        with output:
            print(f"Status Code: {response.status_code}")
            print("Resposta (raw):", response.text)
            try:
                resposta_json = response.json()
                print("Resposta JSON formatada:")
                print(json.dumps(resposta_json, indent=2, ensure_ascii=False))
                if response.status_code == 200 and 'redirectUrl' in resposta_json:
                    # Armazena credenciais e cookies para uso posterior
                    secure_credentials['user'] = user
                    secure_credentials['password'] = password
                    secure_credentials['cookies'] = response.cookies.get_dict()
                    print("\nLogin bem-sucedido. Cookies armazenados para requisições futuras.")
                    login_button.description = 'Logado!'
                    login_button.button_style = 'success'
                    login_button.disabled = True
                else:
                    print("\nFalha no login: verifique usuário e senha.")
            except Exception as e:
                print("Não foi possível interpretar a resposta JSON:", e)
    except Exception as e:
        with output:
            print("Erro na requisição:", e)

login_button.on_click(on_login_clicked)

login_box = widgets.VBox([user_input, password_input, login_button, output])
display(login_box)


In [ ]:
#IMPRIME INVENTÁRIO DA UEP - TESTE COM PARÂMETROS FIXOS

download_button = widgets.Button(description='Baixar PDF', button_style='primary')
output_pdf = widgets.Output()

def baixar_pdf(b):
    cookies = secure_credentials.get('cookies')
    if not cookies:
        with output_pdf:
            clear_output()
            print("Erro: faça login com sucesso antes de tentar baixar o PDF.")
        return
    
    post_data = {
        'idUnidadeExploracao': 78341,
        'dtConsulta': '20/05/2025'
    }
    
    pdf_url = "https://sigen.cidasc.sc.gov.br/DSA.Cadastros/UnidadeExploracao/ImprimeInventarioAnimais"
    
    try:
        pdf_response = requests.post(pdf_url, data=post_data, cookies=cookies)
        with output_pdf:
            clear_output()
            print(f"Status Code: {pdf_response.status_code}")
            content_type = pdf_response.headers.get('Content-Type', '')
            print(f"Content-Type recebido: {content_type}")
            if 'application/pdf' in content_type:
                with open("inventario_animais.pdf", "wb") as f:
                    f.write(pdf_response.content)
                print("PDF baixado com sucesso: inventario_animais.pdf")
            else:
                with open("resposta_nao_pdf.html", "wb") as f:
                    f.write(pdf_response.content)
                print("A resposta não é PDF. Conteúdo salvo em 'resposta_nao_pdf.html'.")
    except Exception as e:
        with output_pdf:
            print("Erro ao tentar baixar o PDF:", e)

download_button.on_click(baixar_pdf)

display(widgets.VBox([download_button, output_pdf]))



In [ ]:
# TESTE BUSCA UEPS

import ipywidgets as widgets
from IPython.display import display, clear_output
import requests
from urllib.parse import urlencode

# Input para o código oficial da propriedade
codigo_input = widgets.Text(
    value='79596',
    description='Código Oficial:',
    placeholder='Digite o código oficial'
)

# Botão de busca
buscar_uep_button = widgets.Button(description="Buscar UEPs", button_style='primary')
uep_output = widgets.Output()

def on_buscar_uep(b):
    with uep_output:
        clear_output()
        print("Enviando requisição para buscar Unidades de Exploração...")

        # Verifica se os cookies estão armazenados
        cookies = secure_credentials.get('cookies')
        if not cookies:
            print("Erro: cookies de sessão não encontrados. Faça o login primeiro.")
            return

        # Dados do formulário
        post_data = {
            "filtroDataSaida": "",
            "fitroHabilitacao": "false",
            "flTek": "false",
            "filtroEvento": "false",
            "listarExcluidas": "true",
            "listarSituacao": "true",
            "listarTodas": "false",
            "filtroForaDoEstado": "N",
            "idPessoaAutorizada": "0",
            "listarProdutor": "false",
            "id_unidade_exploracao": "",
            "cd_oficial_propriedade": codigo_input.value.strip(),
            "nr_unidade_exploracao": "",
            "flUep": "true",
            "ds_flag_Value": "",
            "ds_flag": "",
            "ext-comp-1075_SelIndex": "",
            "cs_situacao_propriedade_Value": "AT",
            "cs_situacao_propriedade": "Ativa",
            "ext-comp-1076_SelIndex": "1",
            "cb_responsavel_Value": "",
            "cb_responsavel": "Documento (CPF/CNPJ) ou Nome e Município",
            "cb_responsavel_SelIndex": "",
            "cb_Produtor_Value": "",
            "cb_Produtor": "",
            "cb_Produtor_SelIndex": "",
            "cb_Evento_Value": "",
            "cb_Evento": "",
            "cb_Evento_SelIndex": "",
            "cb_especie_animal_Value": "1",
            "cb_especie_animal": "BOVINO",
            "searchUepEspecie_SelIndex": "-1",
            "cb_Localidade_Value": "",
            "cb_Localidade": "",
            "cb_Localidade_SelIndex": "",
            "cb_Municipio_Value": "",
            "cb_Municipio": "",
            "cb_Municipio_SelIndex": "",
            "cb_finalidade_criacao_Value": "",
            "cb_finalidade_criacao": "",
            "searchUepFinalidade_SelIndex": "-1"
        }

        url = "https://sigen.cidasc.sc.gov.br/DSA.Cadastros/UnidadeExploracao/PerformSearch"
        headers = {
            "Content-Type": "application/x-www-form-urlencoded; charset=UTF-8",
            "X-Requested-With": "XMLHttpRequest"
        }

        try:
            response = requests.post(
                url,
                headers=headers,
                data=urlencode(post_data),
                cookies=cookies
            )
            print(f"Status Code: {response.status_code}")
            if response.status_code == 200:
                try:
                    json_resp = response.json()
                    print("Resposta JSON:")
                    print(json.dumps(json_resp, indent=2, ensure_ascii=False))
                except Exception as e:
                    print("Resposta não está em formato JSON. Conteúdo bruto:")
                    print(response.text)
            else:
                print("Erro na requisição.")
        except Exception as e:
            print("Erro ao enviar requisição:", e)

buscar_uep_button.on_click(on_buscar_uep)

# Exibir interface
display(widgets.VBox([codigo_input, buscar_uep_button, uep_output]))


